# 01 — Data Quality, Field Audit & Analytical Contract

This notebook is an analytical working paper. It reads the local USAID SCMS extract and writes only anonymized derived outputs.

In [1]:
from pathlib import Path
import sys, os
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'work' / 'python_packages'))
os.environ['MPLBACKEND']='Agg'
from procurement_intelligence.data import load_scms
import pandas as pd
pd.set_option('display.max_columns', 50)

In [2]:
df = load_scms(ROOT/'data/private/scms_delivery_history.csv')
print(f'{df.shape[0]:,} rows × {df.shape[1]:,} fields')
pd.DataFrame({'field': df.columns, 'dtype': df.dtypes.astype(str).values}).head(12)

10,324 rows × 37 fields


,field,dtype
0,record_id,int64
1,Project Code,str
2,PQ #,str
3,PO / SO #,str
4,ASN/DN #,str
5,country,str
6,Managed By,str
7,Fulfill Via,str
8,Vendor INCO Term,str
9,shipment_mode,str


In [3]:
audit = pd.DataFrame({'field': df.columns, 'missing_rate': df.isna().mean(), 'distinct_values': df.nunique()}).sort_values('missing_rate', ascending=False)
audit.head(15)

,field,missing_rate,distinct_values
po_sent_date,po_sent_date,0.555211,895
Vendor INCO Term,Vendor INCO Term,0.523441,7
freight_usd,freight_usd,0.399651,5432
Weight (Kilograms),Weight (Kilograms),0.382797,3388
PQ #,PQ #,0.259686,1236
PQ First Sent to Client Date,PQ First Sent to Client Date,0.259686,763
dosage,dosage,0.168152,54
shipment_mode,shipment_mode,0.034870,4
insurance_usd,insurance_usd,0.027799,6722
ASN/DN #,ASN/DN #,0.000000,7030


In [4]:
pd.read_csv(ROOT/'results/analytical_contract.csv')

,business_concept,status,evidence_and_limit
0,Financial exposure,SUPPORTED,Positive line-item value has complete coverage...
1,Supplier concentration,SUPPORTED,Vendor and line-item value are present; interm...
2,Product/category dependency,SUPPORTED,Item description and value support vendor shar...
3,Comparable unit-price variance,CONDITIONAL,Only exact product-form-pack-country-year coho...
4,Schedule adherence proxy,CONDITIONAL,"Scheduled and delivered dates are populated, b..."
5,Lead-time risk,UNSUPPORTED,PO date coverage is incomplete and source guid...
6,Quality risk,UNSUPPORTED,"No verified quality, rejection, or defect field."
7,Compliance risk,UNSUPPORTED,No verified compliance or contract-performance...


**Decision.** Quality, compliance and lead-time risk are not observable in this extract and are excluded. Schedule adherence remains conditional because the catalog advises against direct lead-time conclusions.